In [ ]:
import json
import os
from tqdm import tqdm
from dotenv import load_dotenv

from elasticsearch import Elasticsearch

load_dotenv()
ES_HOST = os.getenv("ES_HOST", "http://localhost:9200")
INDEX_NAME = os.getenv("INDEX_NAME")

es_client = Elasticsearch('http://localhost:9200')

In [ ]:
with open("data/putin_complete.json", "r") as f:
    speeches = json.load(f)

es_client.indices.delete(index=INDEX_NAME)

mapping = {
    "properties": {
        "id":   {"type": "integer"},
        "title": {"type": "text"},
        "text":  {"type": "text"},
        "date": {"type": "date"},
    }
}

es_client.indices.create(
    index=INDEX_NAME,
    mappings=mapping,
)

index = 0

for doc in tqdm(speeches, "Indexing..."):
    subset_speech = {k: doc[k] for k in doc.keys() if k in ["date", "title", "transcript_filtered"]}
    subset_speech["text"] = subset_speech["transcript_filtered"]
    subset_speech.pop("transcript_filtered")
    index += 1
    es_client.index(index=INDEX_NAME, id=index, document=subset_speech)

In [ ]:
response = es_client.search(
    index=INDEX_NAME,
    query={
            "range": {
                "date": {
                    "gte": "2000-01-01",
                    "lte": "2001-12-31"
                }
            }
        }
)

subset = response["hits"]["hits"]

In [ ]:
index = 0
for doc in subset:
    index += 1
    es_client.index(index="test_index", id=index, document=doc["_source"])

In [123]:
rep = es_client.search(index="test", query={"match_all": {}}, size=10000)["hits"]["hits"]